In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
import os
sys.path.append(os.path.abspath(".."))
from core.viz import plot_bar, plot_dynamic_trends, plot_line, plot_corr_triangle, plot_scatter, plot_statistical_strip, plot_heatmap
from core.s3 import S3AssetManager

In [2]:
notebook_name = "okuo__rancimat"
s3 = S3AssetManager(notebook_name=notebook_name)

In [3]:
df = pd.read_excel(
    "../raw/rancimat/Resultado de Rancimat.xlsx",
 sheet_name="Hoja1", 
 skiprows=3
 )


cls_num = [
'INDUCCIÓN 100 °C',
'INIDUCCIÓN 120 °C', 
'DIAS', 'MESES',
'AÑOS',
'Peróxidos', 
'Anisidina',
'TVN'
]
for cl in cls_num:
    df[cl] = pd.to_numeric(df[cl], errors='coerce')

In [4]:
df

,FECHA,PRODUCTO,PLANTA,LOTE,INDUCCIÓN 100 °C,INIDUCCIÓN 120 °C,DIAS,MESES,AÑOS,Peróxidos,Anisidina,TVN
0,2025-10-31,HCH,SIBATÉ,270925200C – 041025207A,0.925,0.555,0.230,0.008,0.001,14.75,22.70,16.87
1,2025-11-24,HVP,AMAGÁ,C141025464 - C151025465 - C161025466 -C1710254...,6.640,1.265,91.670,3.056,0.255,NaN,NaN,NaN
2,2025-11-24,HCH,AMAGÁ,B131025392A - B141025393B - B151025395A - B161...,1.505,0.640,1.250,0.042,NaN,NaN,NaN,NaN
3,2025-11-24,HCH,AMAGÁ,B201025402A - B201025403B -B221025406A - B2310...,4.450,0.850,60.876,2.029,0.169,NaN,NaN,NaN
4,2025-11-24,HCH,SIBATÉ,(041025207B - 111025214C),1.025,0.595,0.287,0.010,0.001,3.68,23.94,5.00
5,2025-11-24,HCH,AMAGÁ,B300925372 A - B011025373 A - B011025374B-- B0...,0.855,0.500,0.233,0.008,0.001,12.30,28.98,2.34
6,2025-11-24,HVP,AMAGÁ,C290925452 - C300925453 - C011025454-- C021025...,15.395,2.435,407.616,13.587,1.132,4.66,10.12,11.36
7,2025-11-24,HCH,AMAGÁ,B061025380 A - B071025382 C - B0810253833 A-- ...,0.820,0.625,0.088,0.003,0.000,18.55,49.73,2.94
8,2025-11-24,HVP,AMAGÁ,C061025459 - C071025460 - C081025461 - C091025...,6.275,1.255,73.080,2.436,0.203,6.40,15.58,9.41
9,2025-11-24,HCH,SIBATÉ,(111025214D - 181025221B),3.640,0.900,20.179,0.673,0.056,5.05,16.47,7.22


In [27]:
datos_agrupados = df.groupby(["PRODUCTO", 'PLANTA']).agg(
    count=('LOTE', "count"),
    hour_induction_100 =("INDUCCIÓN 100 °C", "median"),
    hour_induction_120 =('INIDUCCIÓN 120 °C', "median"),
    time_life_date =("DIAS", "median"),
    peroxidos=('Peróxidos', 'median'),
    anisidina=('Anisidina', 'median'),
    tvn=('TVN', 'median'),
).reset_index()
s3.save_dataframe(datos_agrupados, "okuo__rancimat.csv")
datos_agrupados

,PRODUCTO,PLANTA,count,hour_induction_100,hour_induction_120,time_life_date,peroxidos,anisidina,tvn
0,HCH,AMAGÁ,7,1.5050,0.640,1.250,12.30,28.98,2.94
1,HCH,SIBATÉ,6,1.2725,0.575,3.958,5.05,16.47,7.22
2,HVP,AMAGÁ,6,10.4250,1.850,184.754,6.40,15.58,9.41


In [16]:
df["planta-producto"] = df["PLANTA"] + "-" + df["PRODUCTO"]


In [7]:
for col in [ 'INDUCCIÓN 100 °C', 'DIAS', 'Peróxidos', 'Anisidina','TVN']:
    f = plot_statistical_strip(df, x_col="PRODUCTO", y_col=col,
     title=col, color_map={"HCH": "#1C8074", "HVP": "#666666"}, width=1000, height=400,)
    print(f"{col.replace(' ', '_').lower()}_box.html")
    s3.save_plotly_html(f, f"{col.replace(' ', '_').lower()}_box.html")
    f.show()


inducción_100_°c_box.html


dias_box.html


peróxidos_box.html


anisidina_box.html


tvn_box.html


In [24]:
df["planta-producto"].unique()

array(['SIBATÉ-HCH', 'AMAGÁ-HVP', 'AMAGÁ-HCH'], dtype=object)

In [25]:
f = plot_statistical_strip(df[df["DIAS"]<2000], x_col="planta-producto", y_col='DIAS',
     title='DIAS', width=1000, height=400, category_order=['SIBATÉ-HCH', 'AMAGÁ-HCH', 'AMAGÁ-HVP'])
f.show()
s3.save_plotly_html(f,  "tv_planta_productos.html")

In [17]:
df

,FECHA,PRODUCTO,PLANTA,LOTE,INDUCCIÓN 100 °C,INIDUCCIÓN 120 °C,DIAS,MESES,AÑOS,Peróxidos,Anisidina,TVN,planta-producto
0,2025-10-31,HCH,SIBATÉ,270925200C – 041025207A,0.925,0.555,0.230,0.008,0.001,14.75,22.70,16.87,SIBATÉ-HCH
1,2025-11-24,HVP,AMAGÁ,C141025464 - C151025465 - C161025466 -C1710254...,6.640,1.265,91.670,3.056,0.255,NaN,NaN,NaN,AMAGÁ-HVP
2,2025-11-24,HCH,AMAGÁ,B131025392A - B141025393B - B151025395A - B161...,1.505,0.640,1.250,0.042,NaN,NaN,NaN,NaN,AMAGÁ-HCH
3,2025-11-24,HCH,AMAGÁ,B201025402A - B201025403B -B221025406A - B2310...,4.450,0.850,60.876,2.029,0.169,NaN,NaN,NaN,AMAGÁ-HCH
4,2025-11-24,HCH,SIBATÉ,(041025207B - 111025214C),1.025,0.595,0.287,0.010,0.001,3.68,23.94,5.00,SIBATÉ-HCH
5,2025-11-24,HCH,AMAGÁ,B300925372 A - B011025373 A - B011025374B-- B0...,0.855,0.500,0.233,0.008,0.001,12.30,28.98,2.34,AMAGÁ-HCH
6,2025-11-24,HVP,AMAGÁ,C290925452 - C300925453 - C011025454-- C021025...,15.395,2.435,407.616,13.587,1.132,4.66,10.12,11.36,AMAGÁ-HVP
7,2025-11-24,HCH,AMAGÁ,B061025380 A - B071025382 C - B0810253833 A-- ...,0.820,0.625,0.088,0.003,0.000,18.55,49.73,2.94,AMAGÁ-HCH
8,2025-11-24,HVP,AMAGÁ,C061025459 - C071025460 - C081025461 - C091025...,6.275,1.255,73.080,2.436,0.203,6.40,15.58,9.41,AMAGÁ-HVP
9,2025-11-24,HCH,SIBATÉ,(111025214D - 181025221B),3.640,0.900,20.179,0.673,0.056,5.05,16.47,7.22,SIBATÉ-HCH


In [8]:
plot_corr_triangle

<function core.viz.plot_corr_triangle(df: pandas.core.frame.DataFrame, value_cols: Union[str, List[str]], method: str = 'pearson', width: int = 1000, height: int = 600, title: Optional[str] = None, xaxis_title: Optional[str] = None, yaxis_title: Optional[str] = None, output_path: Optional[str] = None, custom_colors: Optional[List[str]] = ['#1C8074', '#666666', '#E4572E', '#29B6F6', '#FFA726']) -> plotly.graph_objs._figure.Figure>

In [9]:
for p in df.PRODUCTO.unique():
    f = plot_corr_triangle(df[df.PRODUCTO==p],
     value_cols=["INDUCCIÓN 100 °C", "INIDUCCIÓN 120 °C", "DIAS", "Peróxidos", "Anisidina", "TVN"],
      title=f"<b> Correlación  {p} </b>", 
    width=1000, height=300)
    print(f"correlacion_{p}.html")
    f.show()
    s3.save_plotly_html(f, f"correlacion_{p}.html")

correlacion_HCH.html


correlacion_HVP.html


In [10]:
f = plot_scatter(df, x_col='Anisidina', y_col='INDUCCIÓN 100 °C', group_col='PRODUCTO')
f.show()

In [11]:
import plotly.graph_objects as go
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
from typing import Optional

def plot_exponential_fit(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    title: str = "Análisis de Degradación",
    xaxis_title: str = "Variable X",
    yaxis_title: str = "Variable Y",
    outlier_threshold_y: Optional[float] = None,
    width: int = 1000,
    height: int = 600
) -> go.Figure:
    """
    Genera un gráfico de dispersión con ajuste de curva exponencial (Decay),
    destacando outliers y mostrando métricas de ajuste (R² y ecuación).

    Args:
        df (pd.DataFrame): DataFrame con los datos.
        x_col (str): Nombre de la columna X (ej. 'Anisidina').
        y_col (str): Nombre de la columna Y (ej. 'Induccion_100C').
        title (str): Título del gráfico.
        xaxis_title (str): Etiqueta eje X.
        yaxis_title (str): Etiqueta eje Y.
        outlier_threshold_y (float, optional): Valor límite en Y. Puntos por encima 
            se considerarán outliers y se excluirán del ajuste matemático.
        width/height (int): Dimensiones del gráfico.

    Returns:
        go.Figure: Objeto Plotly interactivo.
    """

    # --- 1. Preparación de Datos ---
    data = df[[x_col, y_col]].dropna()
    x = data[x_col].values
    y = data[y_col].values

    # Lógica de Filtrado (Outliers vs Datos Limpios)
    if outlier_threshold_y is not None:
        mask_clean = y < outlier_threshold_y
        x_clean, y_clean = x[mask_clean], y[mask_clean]
        x_out, y_out = x[~mask_clean], y[~mask_clean]
    else:
        x_clean, y_clean = x, y
        x_out, y_out = [], []

    # --- 2. Modelo Matemático (Exponencial con Asíntota) ---
    def modelo_exponencial(x_val, a, b, c):
        # y = a * e^(-b * x) + c
        return a * np.exp(-b * x_val) + c

    # Ajuste de curva (Curve Fitting)
    try:
        # Puntos iniciales sugeridos para cinéticas de degradación
        popt, _ = curve_fit(modelo_exponencial, x_clean, y_clean, p0=[5, 0.1, 0.5], maxfev=10000)
        
        # Generar línea suave para el gráfico
        x_line = np.linspace(x.min(), x.max(), 100)
        y_line = modelo_exponencial(x_line, *popt)
        
        # Calcular R²
        y_calc = modelo_exponencial(x_clean, *popt)
        r2 = r2_score(y_clean, y_calc)
        
        # Texto de la ecuación
        eq_text = f"<b>Modelo:</b> y = {popt[0]:.2f}e<sup>-{popt[1]:.2f}x</sup> + {popt[2]:.2f}<br><b>R²:</b> {r2:.3f}"
        fit_success = True
    except Exception as e:
        print(f"No se pudo ajustar el modelo: {e}")
        fit_success = False
        x_line, y_line = [], []
        eq_text = "Ajuste no convergente"

    # --- 3. Construcción del Gráfico (Plotly) ---
    fig = go.Figure()

    # Trace A: Datos "Limpios" (Usados para el modelo)
    fig.add_trace(go.Scatter(
        x=x_clean, y=y_clean,
        mode='markers',
        name='Datos Procesados',
        marker=dict(color='#5F8D8B', size=12, line=dict(width=1, color='white')),
        hovertemplate=f"{xaxis_title}: %{{x}}<br>{yaxis_title}: %{{y}}<extra></extra>"
    ))

    # Trace B: Outliers (Si existen) - Visualmente distintos
    if len(x_out) > 0:
        fig.add_trace(go.Scatter(
            x=x_out, y=y_out,
            mode='markers',
            name='Outliers (Excluidos)',
            marker=dict(color='#EF553B', symbol='x', size=10, line=dict(width=2)),
            hovertemplate=f"<b>OUTLIER</b><br>{xaxis_title}: %{{x}}<br>{yaxis_title}: %{{y}}<extra></extra>"
        ))

    # Trace C: Línea de Tendencia
    if fit_success:
        fig.add_trace(go.Scatter(
            x=x_line, y=y_line,
            mode='lines',
            name='Ajuste Exponencial',
            line=dict(color='#FF5733', width=3, dash='dash'),
            hoverinfo='skip'
        ))

    # --- 4. Layout Corporativo ---
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor='center', font=dict(size=18)),
        width=width,
        height=height,
        template="plotly_white",
        showlegend=True,
        legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99, bgcolor="rgba(255,255,255,0.8)"),
        margin=dict(l=60, r=40, t=80, b=60),
        font=dict(family="Inter, Arial, sans-serif", color="#1F2937"),
        hovermode="closest"
    )

    # Ejes
    fig.update_xaxes(title=xaxis_title, showgrid=True, gridcolor='#E5E7EB', zeroline=False)
    fig.update_yaxes(title=yaxis_title, showgrid=True, gridcolor='#E5E7EB', zeroline=False)

    # Anotación con la Ecuación (Estilo tarjeta flotante)
    if fit_success:
        fig.add_annotation(
            x=0.98, y=0.05,
            xref="paper", yref="paper",
            text=eq_text,
            showarrow=False,
            align="right",
            bgcolor="#F3F4F6",
            bordercolor="#D1D5DB",
            borderwidth=1,
            borderpad=10,
            font=dict(size=12, color="#374151")
        )

    return fig

In [12]:
cond1 = df["PRODUCTO"] == "HCH"
cond2 = df["Anisidina"] > 10
df_hch = df[cond1 & cond2]

fig = plot_exponential_fit(
    df=df_hch,
    x_col='Anisidina',
    y_col='DIAS',
    title="<b>Tiempo vida media por Anisidina en HCH</b>",
    xaxis_title="Anisidina (Oxidación Secundaria)",
    yaxis_title="Tiempo de vida media (Días)",
    outlier_threshold_y=20  # Aquí aplicamos el filtro de "Outlier > 3.0"
)
s3.save_plotly_html(fig, "tiempo_vida_media_por_anisidina_en_hch.html")
fig.show()

In [45]:
df_

,FECHA,PRODUCTO,PLANTA,LOTE,INDUCCIÓN 100 °C,INIDUCCIÓN 120 °C,DIAS,MESES,AÑOS,Peróxidos,Anisidina,TVN
1,2025-11-24,HVP,AMAGÁ,C141025464 - C151025465 - C161025466 -C1710254...,6.640,1.265,91.670,3.056,0.255,NaN,NaN,NaN
6,2025-11-24,HVP,AMAGÁ,C290925452 - C300925453 - C011025454-- C021025...,15.395,2.435,407.616,13.587,1.132,4.66,10.12,11.36
8,2025-11-24,HVP,AMAGÁ,C061025459 - C071025460 - C081025461 - C091025...,6.275,1.255,73.080,2.436,0.203,6.40,15.58,9.41
12,2025-12-12,HVP,AMAGÁ,C271025475 - C281025476-C291025477 - C30102547...,6.390,1.050,148.040,4.935,0.411,17.69,17.75,6.86
14,2025-12-12,HVP,AMAGÁ,C201025469 - C211025470 -C221025471 - C2310254...,14.210,2.615,221.468,7.382,0.615,NaN,NaN,NaN
17,2026-01-07,HVP,AMAGÁ,C041125481 - C051125482 -C061125483 - C0711254...,21.095,8.320,233.510,7.784,0.649,NaN,NaN,NaN


In [13]:
cond1 = df["PRODUCTO"] == "HVP"
df_hvp = df[cond1]
fig = plot_exponential_fit(
    df=df_hvp,
    x_col='Anisidina',
    y_col='MESES',
    title="<b>Tiempo vida media por Anisidina en HVP</b>",
    xaxis_title="Anisidina (Oxidación Secundaria)",
    yaxis_title="Tiempo de vida media (Meses)",
    outlier_threshold_y=500  # Aquí aplicamos el filtro de "Outlier > 3.0"
)
s3.save_plotly_html(fig, "tiempo_vida_media_por_anisidina_en_hvp.html")

fig.show()

/var/folders/1g/77kw2x4j5678s_87_sqc1fpc0000gp/T/ipykernel_60880/3428937534.py:58: OptimizeWarning:

Covariance of the parameters could not be estimated



In [14]:


fig = plot_exponential_fit(
    df=df_hch,
    x_col='INDUCCIÓN 100 °C',
    y_col='TVN',
    title="<b>Tiempo vida media por Anisidina en HVP</b>",
    xaxis_title="Anisidina (Oxidación Secundaria)",
    yaxis_title="Tiempo de vida media (Meses)",
    outlier_threshold_y=500  # Aquí aplicamos el filtro de "Outlier > 3.0"
)

fig.show()

In [15]:
df_hch

,FECHA,PRODUCTO,PLANTA,LOTE,INDUCCIÓN 100 °C,INIDUCCIÓN 120 °C,DIAS,MESES,AÑOS,Peróxidos,Anisidina,TVN
0,2025-10-31,HCH,SIBATÉ,270925200C – 041025207A,0.925,0.555,0.230,0.008,0.001,14.75,22.70,16.87
4,2025-11-24,HCH,SIBATÉ,(041025207B - 111025214C),1.025,0.595,0.287,0.010,0.001,3.68,23.94,5.00
5,2025-11-24,HCH,AMAGÁ,B300925372 A - B011025373 A - B011025374B-- B0...,0.855,0.500,0.233,0.008,0.001,12.30,28.98,2.34
7,2025-11-24,HCH,AMAGÁ,B061025380 A - B071025382 C - B0810253833 A-- ...,0.820,0.625,0.088,0.003,0.000,18.55,49.73,2.94
9,2025-11-24,HCH,SIBATÉ,(111025214D - 181025221B),3.640,0.900,20.179,0.673,0.056,5.05,16.47,7.22
11,2025-12-12,HCH,AMAGÁ,B271025414A - B281025415A -B281025415B - B2910...,1.295,0.500,1.509,0.050,0.004,5.65,17.60,11.14
15,2025-12-12,HCH,SIBATÉ,(241025227 D - 311025234 E),1.520,0.390,7.402,0.247,0.021,2.51,16.44,14.61
